# OCR result review

Notebook para validar se o OCR extraiu texto util, nao apenas se a pipeline executou sem erro.

Ele le os artefatos em `data/processed/ocr`, mostra metricas tecnicas e permite inspecionar imagem original, imagem pre-processada e texto extraido lado a lado.

## 1. Setup

Execute esta celula primeiro. Ela localiza a raiz do projeto e configura a visualizacao das tabelas.

In [ ]:
from pathlib import Path
import json
import textwrap

import pandas as pd
from PIL import Image

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(value):
        print(value)

    def Markdown(value):
        return value


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").exists() and (candidate / "documentation").exists():
            return candidate
    raise RuntimeError("Nao encontrei a raiz do projeto. Abra o notebook dentro do repo sherlock-holmes.")


ROOT = find_project_root()
OCR_ROOT = ROOT / "data" / "processed" / "ocr"
INTERIM_ROOT = ROOT / "data" / "interim" / "ocr"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

print(f"Project root: {ROOT}")
print(f"OCR outputs:  {OCR_ROOT}")
print(f"Interim imgs:  {INTERIM_ROOT}")

## 2. Carregar resultados

Esta celula carrega todos os `summary.csv` ja gerados pelo runner.

In [ ]:
def load_summaries():
    frames = []
    if not OCR_ROOT.exists():
        return pd.DataFrame()

    for summary_path in sorted(OCR_ROOT.rglob("summary.csv")):
        relative_parts = summary_path.relative_to(OCR_ROOT).parts
        run_id = relative_parts[0]
        frame = pd.read_csv(summary_path)
        frame.insert(0, "run_id", run_id)
        frame["summary_path"] = str(summary_path.relative_to(ROOT))
        frames.append(frame)

    if not frames:
        return pd.DataFrame()

    results = pd.concat(frames, ignore_index=True)
    numeric_columns = ["elapsed_seconds", "text_length", "word_count"]
    for column in numeric_columns:
        if column in results.columns:
            results[column] = pd.to_numeric(results[column], errors="coerce")
    return results


results = load_summaries()
if results.empty:
    display(Markdown("**Nenhum resultado encontrado ainda.** Rode o runner de OCR para gerar `summary.csv`."))
else:
    columns = [
        "run_id",
        "tool",
        "preset",
        "category",
        "input_path",
        "status",
        "elapsed_seconds",
        "text_length",
        "word_count",
        "error",
    ]
    display(results[[column for column in columns if column in results.columns]])

## 3. Metricas tecnicas

Estas metricas ainda nao dizem qualidade final, mas ajudam a ver se a execucao foi estavel e se saiu texto.

In [ ]:
if results.empty:
    display(Markdown("Sem resultados para agregar."))
else:
    metrics = (
        results.assign(success=results["status"].eq("success"))
        .groupby(["run_id", "tool", "preset"], dropna=False)
        .agg(
            total_documents=("status", "size"),
            success_count=("success", "sum"),
            avg_elapsed_seconds=("elapsed_seconds", "mean"),
            avg_text_length=("text_length", "mean"),
            avg_word_count=("word_count", "mean"),
            errors=("error", lambda values: sum(bool(str(value).strip()) and str(value).lower() != "nan" for value in values)),
        )
        .reset_index()
    )
    metrics["success_rate"] = metrics["success_count"] / metrics["total_documents"]
    display(metrics.sort_values(["success_rate", "avg_text_length"], ascending=[False, False]))

## 4. Revisao visual de um resultado

Use `review_result(...)` para abrir um resultado especifico. O objetivo e checar se o texto extraido parece util quando comparado com a imagem.

In [ ]:
def resolve_path(path_value):
    path = Path(str(path_value))
    if path.is_absolute():
        return path
    return ROOT / path


def load_result_json(row):
    output_path = resolve_path(row["output_path"])
    return json.loads(output_path.read_text(encoding="utf-8"))


def get_preprocessed_path(row):
    preset = row.get("preset")
    if not preset or preset == "none":
        return None

    input_stem = Path(str(row["input_path"])).stem
    return INTERIM_ROOT / row["run_id"] / preset / row["category"] / f"{input_stem}__{preset}.png"


def display_image(path, title, max_size=(1100, 900)):
    if path is None:
        return

    path = resolve_path(path)
    if not path.exists():
        display(Markdown(f"**{title}:** imagem nao encontrada em `{path}`"))
        return

    image = Image.open(path)
    preview = image.copy()
    preview.thumbnail(max_size)
    rel_path = path.relative_to(ROOT) if path.is_relative_to(ROOT) else path
    display(Markdown(f"**{title}**  \n`{rel_path}`  \nOriginal: `{image.size[0]}x{image.size[1]}` | Preview: `{preview.size[0]}x{preview.size[1]}`"))
    display(preview)


def filter_results(tool=None, preset=None, run_id=None, category=None, status=None, input_path=None):
    filtered = results.copy()
    filters = {
        "tool": tool,
        "preset": preset,
        "run_id": run_id,
        "category": category,
        "status": status,
        "input_path": input_path,
    }
    for column, value in filters.items():
        if value is not None and column in filtered.columns:
            filtered = filtered[filtered[column].eq(value)]
    return filtered.reset_index(drop=True)


def review_result(index=0, tool=None, preset=None, run_id=None, category=None, status="success", input_path=None, max_chars=2500):
    filtered = filter_results(tool=tool, preset=preset, run_id=run_id, category=category, status=status, input_path=input_path)
    if filtered.empty:
        display(Markdown("**Nenhum resultado encontrado com estes filtros.**"))
        return None

    row = filtered.iloc[index]
    payload = load_result_json(row)
    text = payload.get("text", "") or ""

    display(Markdown(f"### {row['tool']} / {row['preset']} / {row['run_id']}"))
    display(pd.DataFrame([row]).T.rename(columns={row.name: "value"}))
    display_image(row["input_path"], "Imagem original")
    display_image(get_preprocessed_path(row), "Imagem pre-processada")

    display(Markdown("**Texto extraido**"))
    print(text[:max_chars])
    if len(text) > max_chars:
        print(f"\n... texto truncado no preview ({len(text)} caracteres no total)")
    return row


In [ ]:
# Exemplo: primeiro resultado real do Tesseract.
# Altere tool/preset/run_id/index para navegar por outros resultados.
review_result(tool="tesseract", preset="none", status="success")

## 5. Comparar OCRs no mesmo documento

Esta visao ajuda a responder: para a mesma imagem, qual ferramenta extraiu o texto mais util?

In [ ]:
def compare_document(input_path=None, category=None, status="success", max_chars=1800):
    if results.empty:
        display(Markdown("Sem resultados carregados."))
        return pd.DataFrame()

    candidates = filter_results(category=category, status=status)
    if candidates.empty:
        display(Markdown("Nenhum candidato encontrado para comparacao."))
        return pd.DataFrame()

    if input_path is None:
        input_path = candidates.iloc[0]["input_path"]

    same_doc = filter_results(input_path=input_path, status=status).sort_values(["tool", "preset", "run_id"])
    if same_doc.empty:
        display(Markdown(f"Nenhum resultado encontrado para `{input_path}`."))
        return pd.DataFrame()

    display(Markdown(f"### Documento comparado: `{input_path}`"))
    display_image(input_path, "Imagem original")

    overview_rows = []
    payloads = []
    for _, row in same_doc.iterrows():
        payload = load_result_json(row)
        text = payload.get("text", "") or ""
        payloads.append((row, text))
        overview_rows.append(
            {
                "run_id": row["run_id"],
                "tool": row["tool"],
                "preset": row["preset"],
                "elapsed_seconds": row.get("elapsed_seconds"),
                "text_length": row.get("text_length"),
                "word_count": row.get("word_count"),
                "preview": " ".join(text.split())[:220],
            }
        )

    overview = pd.DataFrame(overview_rows)
    display(overview)

    for row, text in payloads:
        display(Markdown(f"#### {row['tool']} / {row['preset']} / {row['run_id']}"))
        print(text[:max_chars])
        if len(text) > max_chars:
            print(f"\n... texto truncado no preview ({len(text)} caracteres no total)")

    return overview


compare_document()

## 6. Rascunho de avaliacao humana

Depois de revisar visualmente, use uma coluna simples para marcar qualidade: `good`, `partial`, `bad` ou `failed`.

Isto vira uma metrica humana inicial para comparar ferramentas antes de criarmos um gabarito formal.

In [ ]:
def build_human_review_template(status="success"):
    if results.empty:
        return pd.DataFrame()

    columns = [
        "run_id",
        "tool",
        "preset",
        "category",
        "input_path",
        "status",
        "elapsed_seconds",
        "text_length",
        "word_count",
    ]
    review = filter_results(status=status)[[column for column in columns if column in results.columns]].copy()
    review["human_score"] = ""
    review["human_notes"] = ""
    return review


human_review = build_human_review_template()
display(human_review)


def save_human_review(review_df, filename="documentation/reports/ocr-human-review-draft.csv"):
    output_path = ROOT / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    review_df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")


# Quando preencher human_score/human_notes, rode:
# save_human_review(human_review)